## Imports

In [37]:
# Imports
import pandas as pd
import sklearn
import numpy as np

from sklearn.model_selection import cross_val_score

from sklearn.model_selection import StratifiedKFold

%load_ext autoreload
%autoreload 2
import dataprep as dp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
df = pd.read_csv('data/train_with_target.csv', usecols=['external_code', 'total_sales_6'])
df.columns

Index(['external_code', 'total_sales_6'], dtype='object')

In [39]:
df['strat_key'] = pd.qcut(df['total_sales_6'], 4, labels=False, retbins=False, precision=3, duplicates='raise')

df['strat_key'].value_counts()
df = df.reset_index(drop=True)

df.index

RangeIndex(start=0, stop=5080, step=1)

In [40]:
# Set up StratifiedKFold for validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=7)

fold = np.full(len(df), -1, dtype=int)

for i, (train_index, val_index) in enumerate(skf.split(df, df['strat_key'])):
    fold[val_index] = i

assert (fold == -1).sum() == 0

fold_assignment = pd.DataFrame({
    'external_code': df['external_code'],
    'fold': fold,
})
fold_assignment.to_csv('data/fold_assignment.csv', index=False)

print(fold_assignment['fold'].value_counts().sort_index())
print(fold_assignment['external_code'].nunique(), len(fold_assignment))

df = df.drop(columns=['strat_key'])

fold
0    1016
1    1016
2    1016
3    1016
4    1016
Name: count, dtype: int64
5080 5080


In [41]:
# Get cuts per fold
rows = []

for i in range(n_splits):
    is_train = fold != i
    cuts = dp.derive_cuts(df.loc[is_train, 'total_sales_6'])
    rows.append([i, *cuts])

fold_thresholds = pd.DataFrame(rows, columns=['fold', *dp.quantiles.keys()])
fold_thresholds.to_csv('data/fold_thresholds.csv', index=False)

full_cuts = dp.derive_cuts(df['total_sales_6'])

print(fold_thresholds.round(4).to_string(index=False))
print('All 5080:', np.round(full_cuts, 4))

dev = 100 * np.abs(fold_thresholds[list(dp.quantiles)].to_numpy() / full_cuts - 1)
print('Max % deviation per cut:', np.round(dev.max(axis=0), 3))

 fold  outer_low  inner_low  inner_high  outer_high
    0     0.1239     0.1728      0.2948      0.3850
    1     0.1239     0.1728      0.2930      0.3850
    2     0.1239     0.1737      0.2948      0.3850
    3     0.1239     0.1728      0.2939      0.3852
    4     0.1239     0.1737      0.2948      0.3850
All 5080: [0.1239 0.1728 0.2939 0.385 ]
Max % deviation per cut: [0.    0.543 0.319 0.061]


In [42]:
cut_cols = list(dp.quantiles)
thresholds = fold_thresholds.set_index('fold')

frames = []

for i in range(n_splits):
    cuts = thresholds.loc[i, cut_cols].to_numpy()
    zones = dp.apply_cuts(df['total_sales_6'], cuts)

    f = pd.DataFrame({
        'external_code': df['external_code'].to_numpy(),
        'fold': i,
        'zone': zones,
        'split': np.where(fold == i, 'val', 'train')    
    })

    f['label'] = np.where(
        f['split'] == 'val',
        f['zone'].map(dp.evaluation_map),
        f['zone'].map(dp.train_map)
    )

    f['in_buffer'] = f['zone'].isin([1, 3])

    f = f.dropna(subset=['label'])
    f['label'] = f['label'].astype(int)

    frames.append(f)

fold_labels = pd.concat(frames, ignore_index=True)
fold_labels = fold_labels[['external_code', 'fold', 'split', 'label', 'in_buffer']]    
fold_labels.to_csv('data/fold_labels.csv', index=False)

In [43]:
print(fold_labels.groupby(['fold', 'split']).size().unstack())
print(fold_labels[fold_labels.split == 'train'].groupby(['fold', 'label']).size().unstack())
print(fold_labels[fold_labels.split == 'val'].groupby(['fold', 'label']).size().unstack())

split  train   val
fold              
0       3053  1016
1       3048  1016
2       3051  1016
3       3056  1016
4       3053  1016
label     0     1     2
fold                   
0      1016  1021  1016
1      1017  1015  1016
2      1022  1013  1016
3      1022  1018  1016
4      1016  1021  1016
label    0    1    2
fold                
0      253  509  254
1      252  509  255
2      256  506  254
3      256  507  253
4      253  509  254
